# EPred — Exploratory Data Analysis
## Appliances Energy Prediction (UCI Dataset)

This notebook explores the UCI Appliances Energy Prediction dataset, investigates the distribution of energy usage, visualizes indoor/outdoor environmental correlations, and demonstrates the lag/rolling feature engineering logic.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path(".").resolve().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_dataset
from src.preprocessing import PreprocessingPipeline
from src.config import RAW_DATA_PATH

sns.set_theme(style="whitegrid")
%matplotlib inline

### 1. Load and Inspect Raw Dataset

In [ ]:
df_raw, report = load_raw_dataset(RAW_DATA_PATH)
print("Dataset Shape:", df_raw.shape)
print("Diagnostics Report:", report)
df_raw.head()

### 2. Appliances Consumption Distribution & Outlier Diagnostics

In [ ]:
pipeline = PreprocessingPipeline(RAW_DATA_PATH)
outlier_diag = pipeline.investigate_outliers(df_raw, "Appliances")
print("Outlier diagnostics:", outlier_diag)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_raw["Appliances"], bins=50, kde=True, ax=ax1, color="#1f77b4")
ax1.set_title("Distribution of Appliance Energy (Wh)")
ax1.set_xlabel("Appliances (Wh)")

sns.boxplot(x=df_raw["Appliances"], ax=ax2, color="#aec7e8")
ax2.set_title("Boxplot of Appliance Energy (Wh)")
plt.tight_layout()
plt.show()

### 3. Chronological Energy Consumption Profile

In [ ]:
plt.figure(figsize=(15, 5))
# Plot 1 week subset for clear visual inspection
subset = df_raw.iloc[500:1508]
plt.plot(subset["date"], subset["Appliances"], color="#1f77b4", label="Appliances (Wh)")
plt.title("Weekly Profile: 10-Minute Appliance Energy Consumption", fontsize=13)
plt.ylabel("Appliances (Wh)")
plt.xlabel("Date")
plt.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 4. Correlation Analysis: Indoor Environmental vs Energy Consumption

In [ ]:
sensor_cols = ["Appliances", "lights", "T1", "RH_1", "T2", "RH_2", "T3", "RH_3", "T6", "RH_6", "T_out", "RH_out"]
corr = df_raw[sensor_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Matrix: Sensors and Energy Consumption", fontsize=13)
plt.tight_layout()
plt.show()

### 5. Feature Engineering & Target Preparation

In [ ]:
df_prep, meta = pipeline.prepare_dataset()
print(f"Prepared dataset: {df_prep.shape[0]} rows, {df_prep.shape[1]} columns.")
print("Target column: Appliances_Next_10min")
df_prep[["date", "Appliances", "Appliances_Lag_1", "Appliances_Rolling_Mean_3", "Appliances_Next_10min"]].head()